# Single Stock Risk Report

Parameterized report for one ticker over a date range, built on the cleaned table `ohlc_hw_cleaned` (created by `01_data_cleaning.ipynb`). The cell below is tagged `parameters`, so papermill can override its values.

Returns, volatility, drawdown and Sharpe ratio use the adjusted close and daily log returns. Fractions are kept in calculations; percentages are only used for display.

In [ ]:
# Parameters
ticker = "AAPL"  # Stock ticker to analyze
start_date = "2024-01-01"  # Analysis start date
end_date = "2024-12-31"  # Analysis end date
db_path = "data/data.db"  # Local working copy of the course database
risk_free_rate = 0.02  # Annual risk-free rate for the Sharpe ratio
vol_window = 20  # Rolling volatility window (trading days)
ma_window = 50  # Moving-average window (trading days)

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scrapbook as sb
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

TRADING_DAYS = 252
GREEN, RED, GRAY, NAVY = "#2e8b57", "#c0392b", "#9e9e9e", "#1f3b73"
plt.rcParams.update({"axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 11})

## Data loading

In [ ]:
def load_prices(db: str, symbol: str, start: str, end: str) -> pd.DataFrame:
    """Load cleaned daily prices for one ticker between start and end (inclusive), sorted by date."""
    start_ts, end_ts = pd.Timestamp(start), pd.Timestamp(end)
    if start_ts > end_ts:
        raise ValueError(f"start_date {start} is after end_date {end}")
    if not Path(db).exists():
        raise FileNotFoundError(f"Database not found: {Path(db).resolve()}")
    with sqlite3.connect(db) as conn:
        known = pd.read_sql_query("SELECT DISTINCT ticker FROM ohlc_hw_cleaned", conn)["ticker"]
        if symbol not in set(known):
            raise ValueError(f"Unknown ticker {symbol!r}: not in ohlc_hw_cleaned")
        frame = pd.read_sql_query(
            "SELECT ts, open, high, low, close, adj_close, volume FROM ohlc_hw_cleaned "
            "WHERE ticker = ? AND ts BETWEEN ? AND ? ORDER BY ts",
            conn, params=(symbol, start_ts.strftime("%Y-%m-%d"), end_ts.strftime("%Y-%m-%d")),
        )
    frame["ts"] = pd.to_datetime(frame["ts"])
    return frame.set_index("ts")


data = load_prices(db_path, ticker, start_date, end_date)
min_prices = vol_window + 1  # 20 returns need 21 prices
if len(data) < min_prices:
    raise ValueError(f"Insufficient history for {ticker} between {start_date} and {end_date}: "
                     f"{len(data)} prices, at least {min_prices} needed")

first_obs, last_obs = data.index[0].date(), data.index[-1].date()
# first and last trading dates of the whole database inside the window (weekends and holidays are not gaps)
with sqlite3.connect(db_path) as conn:
    window_first, window_last = conn.execute(
        "SELECT MIN(ts), MAX(ts) FROM ohlc_hw_cleaned WHERE ts BETWEEN ? AND ?",
        (pd.Timestamp(start_date).strftime("%Y-%m-%d"), pd.Timestamp(end_date).strftime("%Y-%m-%d"))).fetchone()
print(f"{ticker} from {start_date} to {end_date}: {len(data)} trading days")
print(f"Observed coverage: {first_obs} to {last_obs}")
if str(first_obs) > window_first or str(last_obs) < window_last:
    print(f"Note: other stocks trade from {window_first} to {window_last} in this window; {ticker}'s supplied "
          "history covers less, so all metrics below use its observed dates only.")

In [ ]:
pd.concat([data.head(3), data.tail(3)])

## Risk metrics

In [ ]:
def calculate_returns(prices: pd.Series) -> pd.Series:
    """Calculate daily log returns: log(P_t / P_{t-1}). The first value is NaN."""
    return np.log(prices / prices.shift(1))


def calculate_volatility(returns: pd.Series, window: int = 20) -> pd.Series:
    """Calculate annualized rolling volatility: rolling std of daily returns times sqrt(252).

    The first window values are NaN (warm-up) and are left as such.
    """
    return returns.rolling(window).std() * np.sqrt(TRADING_DAYS)


def calculate_max_drawdown(prices: pd.Series) -> pd.Series:
    """Return the per-date drawdown from the running price peak: (P_t - max_{s<=t} P_s) / max_{s<=t} P_s.

    The maximum drawdown of the period is the minimum of this series.
    """
    running_peak = prices.cummax()
    return (prices - running_peak) / running_peak


def calculate_sharpe_ratio(returns: pd.Series, risk_free_rate: float = 0.02) -> float:
    """Calculate the annualized Sharpe ratio (mean * 252 - rf) / (std * sqrt(252)) of daily returns.

    Returns NaN when there are fewer than 2 returns or the volatility is zero.
    """
    r = returns.dropna()
    std = r.std(ddof=1)
    if len(r) < 2 or not np.isfinite(std) or std == 0:
        return float("nan")
    return float((r.mean() * TRADING_DAYS - risk_free_rate) / (std * np.sqrt(TRADING_DAYS)))


def summary_statistics(prices: pd.Series, returns: pd.Series, risk_free_rate: float) -> dict:
    """Whole-period summary metrics as fractions (NaN when undefined)."""
    r = returns.dropna()
    n = len(r)
    up, down = r[r > 0], r[r < 0]
    return {
        "Total Return": prices.iloc[-1] / prices.iloc[0] - 1,
        "Annualized Return": np.exp(r.sum() * TRADING_DAYS / n) - 1 if n else np.nan,
        "Annualized Volatility": r.std(ddof=1) * np.sqrt(TRADING_DAYS) if n > 1 else np.nan,
        "Sharpe Ratio": calculate_sharpe_ratio(r, risk_free_rate),
        "Maximum Drawdown": calculate_max_drawdown(prices).min(),
        "Best Day": r.max() if n else np.nan,
        "Worst Day": r.min() if n else np.nan,
        "Positive Days": (r > 0).sum() / n if n else np.nan,
        "Average Up Day": up.mean() if len(up) else np.nan,
        "Average Down Day": down.mean() if len(down) else np.nan,
    }


prices = data["adj_close"]
returns = calculate_returns(prices)
volatility = calculate_volatility(returns, vol_window)
drawdown = calculate_max_drawdown(prices)
moving_average = prices.rolling(ma_window).mean()
stats = summary_statistics(prices, returns, risk_free_rate)

mean_vol = volatility.mean()
print(f"Daily log returns: {returns.notna().sum()} | rolling volatility estimates: {volatility.notna().sum()}")
print(f"Mean {vol_window}-day volatility: {mean_vol:.2%}")
print(f"Sharpe ratio: {stats['Sharpe Ratio']:.2f}" if np.isfinite(stats["Sharpe Ratio"]) else "Sharpe ratio: N/A")
print(f"Maximum drawdown: {drawdown.min():.2%} on {drawdown.idxmin().date()}")

## Visualization

Price segments are colored by the rolling volatility at their end date compared with its mean over the report: red above the mean, green at or below it, gray during the warm-up period where no estimate exists. This coloring is retrospective, since the mean uses the whole period.

In [ ]:
def plot_price(ax):
    x = mdates.date2num(prices.index.to_pydatetime())
    points = np.column_stack([x, prices.to_numpy()])
    segments = np.stack([points[:-1], points[1:]], axis=1)
    seg_vol = volatility.to_numpy()[1:]
    colors = np.where(np.isnan(seg_vol), GRAY, np.where(seg_vol > mean_vol, RED, GREEN))
    ax.add_collection(LineCollection(segments, colors=colors, linewidths=2))
    ax.autoscale_view()
    handles = [Line2D([], [], color=RED, lw=2, label="High volatility (> mean)"),
               Line2D([], [], color=GREEN, lw=2, label="Low volatility (<= mean)"),
               Line2D([], [], color=GRAY, lw=2, label="Volatility warm-up")]
    if moving_average.notna().any():
        ax.plot(moving_average.index, moving_average, color=NAVY, lw=1.6, ls="--")
        handles.append(Line2D([], [], color=NAVY, lw=1.6, ls="--", label=f"{ma_window}-day MA"))
    else:
        ax.text(0.02, 0.04, f"{ma_window}-day MA unavailable: only {len(prices)} prices",
                transform=ax.transAxes, color=NAVY, fontsize=10)
    ax.legend(handles=handles, loc="best", fontsize=9)
    ax.set(title=f"{ticker} Price and {ma_window}-day MA", xlabel="Date", ylabel="Adjusted close (USD)")


def plot_returns(ax):
    r = returns.dropna()
    ax.bar(r.index, r, color=np.where(r >= 0, GREEN, RED), width=1.0)
    ax.axhline(0, color="black", lw=0.8)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set(title=f"{ticker} Daily Returns", xlabel="Date", ylabel="Daily log return")


def plot_volatility(ax):
    v = volatility
    ax.plot(v.index, v, color=NAVY, lw=1.6, label=f"{vol_window}-day volatility")
    ax.axhline(mean_vol, color="black", ls="--", lw=1, label=f"Mean ({mean_vol:.1%})")
    ax.fill_between(v.index, v, mean_vol, where=v > mean_vol, interpolate=True, color=RED, alpha=0.2,
                    label="High volatility")
    ax.fill_between(v.index, v, mean_vol, where=v < mean_vol, interpolate=True, color=GREEN, alpha=0.2,
                    label="Low volatility")
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.legend(loc="best", fontsize=9)
    ax.set(title=f"{ticker} Rolling Volatility ({vol_window}-day)", xlabel="Date", ylabel="Annualized volatility")


def plot_drawdown(ax):
    dd = drawdown
    for scale in (1.0, 0.75, 0.5, 0.25):  # layered transparent fills give a red gradient
        ax.fill_between(dd.index, dd * scale, 0, color=RED, alpha=0.15, lw=0)
    ax.plot(dd.index, dd, color=RED, lw=1.2)
    worst = dd.idxmin()
    ax.scatter([worst], [dd.min()], color="black", zorder=5)
    ax.annotate(f"Max drawdown {dd.min():.1%}\n{worst.date()}", xy=(worst, dd.min()),
                xytext=(10, 10), textcoords="offset points", fontsize=10)
    ax.set_ylim(min(dd.min() * 1.25, -0.01), 0.005)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set(title=f"{ticker} Maximum Drawdown", xlabel="Date", ylabel="Drawdown from peak")


fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plot_price(axes[0, 0])
plot_returns(axes[0, 1])
plot_volatility(axes[1, 0])
plot_drawdown(axes[1, 1])
x_range = mdates.date2num([prices.index[0].to_pydatetime(), prices.index[-1].to_pydatetime()])
for ax in axes.flat:
    ax.grid(True, alpha=0.3)
    ax.set_xlim(x_range[0] - 1, x_range[1] + 1)  # same date axis on all panels
    locator = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
fig.suptitle(f"{ticker} Risk Analysis Report - {start_date} to {end_date}", fontsize=17, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

## Summary statistics

Total return is last over first adjusted close minus 1. With n log returns, the annualized return is `exp(sum * 252 / n) - 1` and the annualized volatility is the whole-period standard deviation times sqrt(252), which differs from the mean rolling volatility. Best and worst days and up/down averages use log returns; positive days are counted over all non-missing returns.

In [ ]:
def fmt(value: float, kind: str) -> str:
    if value is None or not np.isfinite(value):
        return "N/A"
    return f"{value:.2f}" if kind == "ratio" else (f"{value:+.2%}" if kind == "signed" else f"{value:.2%}")


kinds = {"Total Return": "signed", "Annualized Return": "signed", "Annualized Volatility": "pct",
         "Sharpe Ratio": "ratio", "Maximum Drawdown": "signed", "Best Day": "signed", "Worst Day": "signed",
         "Positive Days": "pct", "Average Up Day": "signed", "Average Down Day": "signed"}
summary_table = pd.DataFrame({"Metric": list(stats), "Value": [fmt(stats[k], kinds[k]) for k in stats]})
print(f"{ticker}: observed {first_obs} to {last_obs} ({len(data)} trading days, {returns.notna().sum()} returns)")
print(summary_table.to_string(index=False))

In [ ]:
# Export named results for the batch runner (papermill + scrapbook)
def clean_float(value):
    return float(value) if value is not None and np.isfinite(value) else None


metrics = {
    "ticker": ticker,
    "start_date": start_date,
    "end_date": end_date,
    "first_date": str(first_obs),
    "last_date": str(last_obs),
    "trading_days": int(len(data)),
    "avg_daily_return": clean_float(returns.mean()),
    "avg_volatility": clean_float(mean_vol),
    "max_drawdown": clean_float(drawdown.min()),
    "sharpe_ratio": clean_float(stats["Sharpe Ratio"]),
    "total_return": clean_float(stats["Total Return"]),
    "annualized_return": clean_float(stats["Annualized Return"]),
    "annualized_volatility": clean_float(stats["Annualized Volatility"]),
}
sb.glue("metrics", metrics)